In [ ]:
# ==============================================================================
# CELDA 1: LIBRERÍAS Y PARÁMETROS CERTIFICADOS (ORDINAL - ANTI-OVERFITTING)
# ==============================================================================
import os, random, re, glob, warnings, copy, gc, cv2, types
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

import scipy.stats as st

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, confusion_matrix, cohen_kappa_score, 
                             f1_score, roc_auc_score, roc_curve, precision_recall_curve, auc)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import BayesianRidge
from torch.optim.swa_utils import AveragedModel, SWALR

import matplotlib.pyplot as plt
import seaborn as sns

import warnings


warnings.filterwarnings('ignore')


import numpy as np
import pandas as pd
import scipy.stats as st
from sklearn.linear_model import BayesianRidge
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import (mean_absolute_error, confusion_matrix, cohen_kappa_score, 
                             roc_auc_score, roc_curve, precision_recall_curve, auc)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import warnings
warnings.filterwarnings("ignore")


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap   
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import BayesianRidge 



# Parámetros
DEBUG_MODE = False   
SEED = 13          
N_SPLITS = 5       
BATCH_SIZE = 4      
ACCUM_STEPS = 4     # Batch Efectivo = 16
N_CORTES = 16      

# Hiperparametros
LR_BACKBONE = 1.36e-05
LR_HEAD = 1.53e-04
WEIGHT_DECAY = 5.37e-03
CUTMIX_PROB = 0.0885
SWA_LR_OPT = 1.56e-05

if DEBUG_MODE:
    N_SPLITS = 2; EPOCHS_RUN = 3; SWA_START = 1; ACCUM_STEPS = 1
    print(" WARNING: DEBUG_MODE ACTIVO (Ejecución ultrarrápida de prueba).")
else:
    EPOCHS_RUN = 40; SWA_START = 25   

# Inicialización de semillas
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Entorno Visual Ordinal Certificado. Aceleración: {dispositivo}")

# ---------------------------------------------------------
# AUMENTO MORFOLÓGICO (Simulación de Atrofia Ex-Vacuo)
# ---------------------------------------------------------
class MorphologicalAugmentation:
    def __init__(self, p=0.3, kernel_size=3):
        self.p = p; self.kernel = np.ones((kernel_size, kernel_size), np.uint8)
    def __call__(self, img):
        if random.random() < self.p:
            img_np = np.array(img)
            img_np = cv2.dilate(img_np, self.kernel, iterations=1) if random.random() < 0.5 else cv2.erode(img_np, self.kernel, iterations=1)
            return Image.fromarray(img_np)
        return img

In [ ]:
# ==============================================================================
# CELDA 2: CARGA CLÍNICA, MAPEADO TOPOLÓGICO ORDINAL Y MUESTREO DEBUG
# ==============================================================================
PATH_IMGS = "/kaggle/input/datasets/ninadaithal/imagesoasis"
EXCEL_FILES = glob.glob("/kaggle/input/**/*.xlsx", recursive=True)

df_demog = pd.read_excel(EXCEL_FILES[0])
df_demog['id_paciente'] = df_demog['ID'].str.extract(r'(OAS1_\d{4})')
df_demog = df_demog.drop_duplicates(subset=['id_paciente'], keep='first')

# Filtrado por viabilidad (Edad >= 56)
valid_ids = set(df_demog[df_demog['Age'] >= 56]['id_paciente'].dropna().unique())

archivos = []
for root, _, files in os.walk(PATH_IMGS):
    for f in files:
        if f.lower().endswith(('.png', '.jpg')):
            pid_match = re.search(r"OAS1_\d{4}", f, re.IGNORECASE)
            if pid_match and pid_match.group(0).upper() in valid_ids:
                archivos.append({"ruta": os.path.join(root, f), "id": pid_match.group(0).upper()})

df_raw = pd.DataFrame(archivos)
df_pacientes = df_raw[['id', 'ruta']].drop_duplicates(subset=['id']).reset_index(drop=True)

cols_clinicas = ['id_paciente', 'Age', 'Educ', 'SES', 'MMSE', 'eTIV', 'nWBV', 'CDR']
df_pacientes = pd.merge(df_pacientes, df_demog[cols_clinicas], left_on='id', right_on='id_paciente', how='inner')

# MAPEADO TOPOLÓGICO CONTINUO (Regresión Ordinal Acotada)
df_pacientes = df_pacientes.dropna(subset=['CDR']).reset_index(drop=True) # Purga de NaNs
# Sano=0.0, MCI=0.5, Demencia (1.0 o más) se acota a 1.0 para mantener el Sigmoide
df_pacientes['etiqueta'] = df_pacientes['CDR'].clip(0.0, 1.0) 

# Estratificación Dual para K-Fold
df_pacientes['Age_Quartile'] = pd.qcut(df_pacientes['Age'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
df_pacientes['strat_dual'] = df_pacientes['etiqueta'].astype(str) + "_" + df_pacientes['Age_Quartile'].astype(str)

conteo = df_pacientes['strat_dual'].value_counts()
clases_min = conteo[conteo < N_SPLITS].index
df_pacientes.loc[df_pacientes['strat_dual'].isin(clases_min), 'strat_dual'] = df_pacientes.loc[df_pacientes['strat_dual'].isin(clases_min), 'etiqueta'].astype(str)

# ---------------------------------------------------------
# INTERVENCIÓN DE DEBUG FAILSAFE PARA MULTICLASE
# ---------------------------------------------------------
if DEBUG_MODE:
    print("\n MODO DEBUG ACTIVADO\n")
    # En multiclase necesitamos garantizar representación de los 3 estados en el debug
    muestras_por_estrato = max(N_SPLITS, (BATCH_SIZE * ACCUM_STEPS) // len(df_pacientes['strat_dual'].unique()) + 2)
    df_pacientes = df_pacientes.groupby('strat_dual', group_keys=False).apply(
        lambda x: x.sample(min(len(x), muestras_por_estrato), random_state=SEED)
    ).reset_index(drop=True)

print(f"Total Pacientes: {len(df_pacientes)}")
print(f"Distribución:\n{df_pacientes['etiqueta'].value_counts().sort_index()}")

In [ ]:
# ==============================================================================
# CELDA 3: DATASET VISUAL SOTA (CACHÉ ANATÓMICO PRECOMPUTADO PARA I/O ÓPTIMO)
# ==============================================================================

class MorphologicalAugmentation:
    """
    Aumentación Estocástica de Datos Clínicos.
    En lugar de añadir simple "ruido" a los píxeles, simula biológicamente la 
    enfermedad: usa Erosión para simular pérdida de materia gris (atrofia) y 
    Dilatación para simular agrandamiento de los ventrículos (atrofia ex-vacuo).
    """
    def __init__(self, p=0.3, kernel_size=3):
        self.p = p # Probabilidad de que se aplique la alteración (30%)
        # Matriz de 3x3 píxeles que actuará como 'pincel' para engordar o adelgazar el tejido
        self.kernel = np.ones((kernel_size, kernel_size), np.uint8)
        
    def __call__(self, img):
        if np.random.random() < self.p:
            img_np = np.array(img)
            # Elige al azar si simula un cerebro más sano (dilatación del tejido) 
            # o más enfermo (erosión del tejido)
            img_np = cv2.dilate(img_np, self.kernel, iterations=1) if np.random.random() < 0.5 else cv2.erode(img_np, self.kernel, iterations=1)
            return Image.fromarray(img_np)
        return img

class OASIS_Visual_Dataset(Dataset):
    """
    Dataloader Inteligente con Caché.
    Realiza una segmentación de tejido en tiempo de inicialización para 
    asegurar que los cortes seleccionados siempre contengan masa cerebral útil,
    evitando enviar a la GPU cortes vacíos o puramente craneales.
    """
    def __init__(self, dataframe: pd.DataFrame, is_train: bool = True, n_cortes: int = 16):
        self.rutas = dataframe['ruta'].values
        self.targets = dataframe['etiqueta'].values 
        self.pids = dataframe['id'].values 
        self.is_train = is_train
        self.n_cortes = n_cortes 
        
        # 1. Transformaciones Obligatorias
        self.transform_base = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])
        
        # 2. Transformaciones de Regularización (Solo para entrenamiento)
        # Exclusivamente transformaciones espaciales isométricas (sin flips para mantener asimetría)
        self.augment = transforms.Compose([
            MorphologicalAugmentation(p=0.3),
            transforms.RandomRotation(5), # Rotación muy leve (5 grados)
            transforms.RandomAffine(degrees=0, translate=(0.02, 0.02), scale=(0.98, 1.02))
        ])
        
        self.rutas_precomputadas = {}
        self._precalcular_mapa_anatomico()
        
    def _precalcular_mapa_anatomico(self):
        """
        Heurística de Umbralización de Otsu.
        Busca el corte con mayor área de tejido cerebral y ancla allí la extracción.
        """
        for idx in range(len(self.pids)):
            paciente_id = self.pids[idx]
            carpeta_clase = os.path.dirname(self.rutas[idx])
            
            todos = os.listdir(carpeta_clase)
            # Ordenar es vital para no alterar el eje Z (de arriba a abajo)
            archivos = sorted([f for f in todos if f.lower().endswith(('.png', '.jpg')) and paciente_id.lower() in f.lower()])
            
            if not archivos:
                raise ValueError(f"Fallo estructural: No se encontraron imágenes para {paciente_id}")

            # FASE DE BÚSQUEDA DEL "ECUADOR" DEL CEREBRO
            areas = []
            for f in archivos:
                img_path = os.path.join(carpeta_clase, f)
                # Lee la imagen en blanco y negro (muy rápido)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                
                # Algoritmo de Otsu: Calcula matemáticamente dónde cortar entre el negro del fondo y el gris del cerebro.
                # Crea una máscara binaria (blanco o negro absoluto)
                _, thresh = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                
                # Cuenta los píxeles blancos (área del cerebro) y lo guarda
                areas.append(cv2.countNonZero(thresh))
                
            areas = np.array(areas)
            corte_max_area = np.argmax(areas) # Índice del corte más "gordo"
            
            # FASE DE ANCLAJE
            # En lugar de centrar la ventana justo en el medio, la bajamos un 20%
            # porque los biomarcadores críticos (Hipocampo, Lóbulo Temporal) están por debajo del ecuador.
            idx_start = max(0, corte_max_area - int(self.n_cortes * 0.2))
            idx_end = min(len(archivos) - 1, idx_start + self.n_cortes - 1)
            
            # Control de seguridad: Si nos salimos por arriba, ajustamos por abajo
            if (idx_end - idx_start + 1) < self.n_cortes:
                idx_start = max(0, idx_end - self.n_cortes + 1)
                
            indices_seleccionados = np.linspace(idx_start, idx_end, self.n_cortes).astype(int)
            
            # Guardamos las rutas finales en el diccionario de la RAM (Caché)
            self.rutas_precomputadas[idx] = [os.path.join(carpeta_clase, archivos[i]) for i in indices_seleccionados]

    def __len__(self) -> int: 
        return len(self.rutas)
        
    def __getitem__(self, idx: int):
        rutas_anatomicas = self.rutas_precomputadas[idx]
        
        cortes_tensor = []
        for ruta in rutas_anatomicas:
            img = Image.open(ruta).convert('RGB')
            if self.is_train: 
                img = self.augment(img)
            
            t = self.transform_base(img)
            # Z-Score Instance Normalization (esencial para estabilidad de gradientes)
            t = (t - t.mean()) / (t.std() + 1e-6) 
            cortes_tensor.append(t)
            
        # Shape devuelto: [8, 3, 224, 224] (Batch, Slices, Canales, Alto, Ancho)
        return torch.stack(cortes_tensor), torch.tensor(self.targets[idx]).float()

In [ ]:
# ==============================================================================
# CELDA 4: ARQUITECTURA SOTA ORDINAL (CONVNEXT + SPD-CONV + CORAL + DROPOUT)
# ==============================================================================

class SPDConv(nn.Module):
    """
    Space-to-Depth Convolution. 
    Reduce la dimensionalidad espacial (H, W) empaquetando píxeles en canales (C).
    Evita la pérdida de información anatómica subpíxel típica del Max-Pooling clásico.
    """
    def __init__(self, in_channels, out_channels, block_size=2):
        super().__init__()
        self.block_size = block_size
        self.conv = nn.Conv2d(in_channels * (block_size ** 2), out_channels, kernel_size=3, padding=1, bias=False)
        self.norm = nn.LayerNorm(out_channels)
        
    def forward(self, x):
        B, C, H, W = x.size()
        bs = self.block_size
        pad_h, pad_w = (bs - H % bs) % bs, (bs - W % bs) % bs
        if pad_h > 0 or pad_w > 0: x = F.pad(x, (0, pad_w, 0, pad_h)); H, W = H + pad_h, W + pad_w
        out_H, out_W = H // bs, W // bs
        x = x.view(B, C, out_H, bs, out_W, bs).permute(0, 1, 3, 5, 2, 4).contiguous().view(B, C * (bs ** 2), out_H, out_W)
        return self.norm(self.conv(x).permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

class MAF(nn.Module):
    """Mixed Activation Function: Suaviza la topología de los gradientes (ReLU + SiLU + GELU)."""
    def forward(self, x): return (F.relu(x) + F.silu(x) + F.gelu(x)) / 3.0

class HybridAgile_MIDL(nn.Module):
    """
    Arquitectura Ordinal 2.5D.
    Extrae características con ConvNeXt (Deep Frozen), pondera la relevancia clínica 
    de cada corte (Slice Attention) y emite un pronóstico ordinal (CORAL).
    """
    def __init__(self, num_slices=16, feature_dim=768, hidden_dim=256):
        super().__init__()
        # 1. EXTRACTOR VISUAL UNIVERSAL (ImageNet)
        self.backbone = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        
        # CONGELACIÓN SEVERA (Deep Freezing)
        # En la regresión ordinal con Small Data, el "Moving Target Problem" destruye la convergencia.
        # Congelamos todo el backbone de ConvNeXt para que actúe 
        # como un extractor de texturas determinista y estable.
        for param in self.backbone.parameters(): 
            param.requires_grad = False
            
        self.backbone.classifier = nn.Identity() # Anulamos la cabeza de clasificación original
        
        # 2. CAPAS ENTRENABLES (De aquí en adelante sí hay gradientes)
        self.fine_grained_extractor = SPDConv(in_channels=768, out_channels=feature_dim, block_size=2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # 3. MECANISMO DE ATENCIÓN (Slice-Attention)
        self.attention_V = nn.Sequential(nn.Linear(feature_dim, hidden_dim), MAF())
        self.attention_U = nn.Sequential(nn.Linear(feature_dim, hidden_dim), nn.Sigmoid())
        self.attention_weights = nn.Linear(hidden_dim, 1)
        
        # 4. CUELLO DE BOTELLA Y REGULARIZACIÓN
        # REGULARIZACIÓN AGRESIVA: Aumentamos el Dropout al 60% (p=0.6).
        # Como el modelo ordinal evalúa trayectorias complejas, forzamos a la red 
        # a no depender de ninguna neurona latente específica para evitar memorizar a la clase minoritaria (N=23).
        self.latent_extractor = nn.Sequential(
            nn.Linear(feature_dim, 128), 
            nn.LayerNorm(128), 
            MAF(), 
            nn.Dropout(p=0.6) 
        )
        
        # 5. CABECERA CORAL (Consistent Rank Logits)
        # En vez de N nodos independientes, proyectamos a un único valor continuo (bias=False) que mide la "gravedad"
        self.fc = nn.Linear(128, 1, bias=False)
        # Y creamos K-1 umbrales (Para 3 clases: Sano, MCI, Demencia -> 2 umbrales)
        # Estos sesgos "aprenden" a situarse en los puntos de corte exactos de la enfermedad.
        self.ordinal_bias = nn.Parameter(torch.zeros(2))

    def forward(self, x):
        # x shape: [Batch, Slices, Channels, H, W]
        B, S, C, H, W = x.shape
        x = x.view(B * S, C, H, W) # Truco 2.5D: Fusiona pacientes y cortes
        
        # OPTIMIZACIÓN DE VRAM: Pass sin gradientes por el backbone congelado.
        # Al no guardar el grafo de gradientes de ConvNeXt, ahorramos casi un 50% de memoria gráfica.
        with torch.no_grad():
            features_spatial = self.backbone.features(x) 
            
        features_flat = self.adaptive_pool(self.fine_grained_extractor(features_spatial)).view(B * S, -1)
        
        # Cálculo y aplicación de Pesos de Atención
        A = F.softmax(self.attention_weights(self.attention_V(features_flat) * self.attention_U(features_flat)).view(B, S), dim=1)
        features_flat = features_flat.view(B, S, -1)
        M = torch.bmm(A.unsqueeze(1), features_flat).squeeze(1) # Reagrupación por paciente
        
        # Proyección al espacio latente comprimido (128d)
        latent_128d = self.latent_extractor(M)
        
        # PROYECCIÓN CORAL
        proj = self.fc(latent_128d)              # Vector de gravedad [Batch, 1]
        # Broadcasting: Suma los 2 umbrales al vector de gravedad al mismo tiempo. 
        # Salida: [Batch, 2]. Son los Logits independientes para cada tarea binaria de la cascada.
        logits = proj + self.ordinal_bias        
        return logits, latent_128d, A

print(" Arquitectura instanciada")

In [ ]:
# ==============================================================================
# CELDA 5: MOTOR VISUAL ORDINAL (CORAL FOCAL SMOOTH + SWA)
# ==============================================================================
DIR_GUARDADO = './modelos_entrenados/'
os.makedirs(DIR_GUARDADO, exist_ok=True) 

# 1. TEST-TIME AUGMENTATION (TTA)
tta_transforms_test = transforms.Compose([transforms.RandomAffine(degrees=5, scale=(0.9, 1.0))])
N_TTA = 10 if not DEBUG_MODE else 2 

resultados_visuales_por_fold = {}
historial_train_loss = []
historial_val_loss = []

skf_outer = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

def get_cascade_labels(targets_batch, device):
    """ Convierte etiqueta clínica (0.0, 0.5, 1.0) en cascada binaria [T1, T2] """
    cascade_labels = torch.zeros((targets_batch.size(0), 2), device=device)
    cascade_labels[:, 0] = (targets_batch >= 0.5).float() # Tarea 1: ¿Deterioro?
    cascade_labels[:, 1] = (targets_batch >= 1.0).float() # Tarea 2: ¿Demencia?
    return cascade_labels

def cutmix_data(x, y, alpha=1.0):
    """ Aumento Estocástico Espacial CutMix """
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    index = torch.randperm(x.size(0)).to(x.device)
    W, H = x.size(-1), x.size(-2)
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1, bby1 = np.clip(cx - cut_w // 2, 0, W), np.clip(cy - cut_h // 2, 0, H)
    bbx2, bby2 = np.clip(cx + cut_w // 2, 0, W), np.clip(cy + cut_h // 2, 0, H)
    x[:, :, :, bby1:bby2, bbx1:bbx2] = x[index, :, :, bby1:bby2, bbx1:bbx2]
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (W * H))
    return x, y, y[index], lam

# FOCAL SMOOTH LOSS ADAPTADA A CORAL
class CoralFocalSmoothLoss(nn.Module):
    def __init__(self, pos_weight, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        # Usamos reduction='none' para poder aplicar la focal loss matemáticamente
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction='none')
        self.gamma = gamma
        self.ls = label_smoothing

    def forward(self, logits, targets):
        # 1. Label Smoothing: Amortigua el "choque térmico" al apagar CutMix
        targets_smooth = targets * (1.0 - self.ls) + 0.5 * self.ls
        
        # 2. Pérdida Cruda
        bce_loss = self.bce(logits, targets_smooth)
        
        # 3. Focal Modulator: Suaviza los picos de pérdida del Batch Size pequeño
        pt = torch.exp(-bce_loss) 
        f_loss = ((1 - pt)**self.gamma * bce_loss).mean()
        
        return f_loss

# ==============================================================================
# BUCLE PRINCIPAL (K-FOLD)
# ==============================================================================
for fold_out, (train_val_idx, test_idx) in enumerate(skf_outer.split(df_pacientes, df_pacientes['strat_dual'])):
    print(f"\n FOLD {fold_out + 1}/{N_SPLITS}")
    
    df_train_val = df_pacientes.iloc[train_val_idx].copy()
    df_test_outer = df_pacientes.iloc[test_idx].copy()
    
    min_clase = df_train_val['strat_dual'].value_counts().min()
    if min_clase < 2:
        indices_tr = np.arange(len(df_train_val)); np.random.shuffle(indices_tr)
        split_idx = max(1, int(len(indices_tr) * 0.8))
        train_idx_inner, val_idx_inner = indices_tr[:split_idx], indices_tr[split_idx:]
    else:
        n_splits_inner = min(5, min_clase)
        skf_split = StratifiedKFold(n_splits=n_splits_inner, shuffle=True, random_state=SEED) 
        train_idx_inner, val_idx_inner = next(skf_split.split(df_train_val, df_train_val['strat_dual']))
    
    loader_tr = DataLoader(OASIS_Visual_Dataset(df_train_val.iloc[train_idx_inner], is_train=True, n_cortes=N_CORTES), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    loader_va = DataLoader(OASIS_Visual_Dataset(df_train_val.iloc[val_idx_inner], is_train=False, n_cortes=N_CORTES), batch_size=BATCH_SIZE, num_workers=2)
    loader_te = DataLoader(OASIS_Visual_Dataset(df_test_outer, is_train=False, n_cortes=N_CORTES), batch_size=BATCH_SIZE, num_workers=2) 

    model = HybridAgile_MIDL().to(dispositivo)
    swa_model = AveragedModel(model) 
    scaler_amp = torch.cuda.amp.GradScaler() 
    
    pesos_clases = torch.tensor([3.0, 4.5]).to(dispositivo) 
    crit_coral = CoralFocalSmoothLoss(pos_weight=pesos_clases, gamma=1.5, label_smoothing=0.1).to(dispositivo)
    
    m = model.module if isinstance(model, nn.DataParallel) else model
    
    head_params = list(m.fine_grained_extractor.parameters()) + \
                  list(m.attention_V.parameters()) + \
                  list(m.attention_U.parameters()) + \
                  list(m.attention_weights.parameters()) + \
                  list(m.latent_extractor.parameters()) + \
                  list(m.fc.parameters()) + [m.ordinal_bias]
                  
    for p in head_params: p.requires_grad = True
        
    wd_agressive = WEIGHT_DECAY * 10 
    param_groups = [{'params': head_params, 'lr': LR_HEAD, 'weight_decay': wd_agressive}]
    opt = torch.optim.AdamW(param_groups) 
    
    pasos = max(1, len(loader_tr) // ACCUM_STEPS)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=pasos*10, T_mult=2, eta_min=1e-6)
    swa_scheduler = SWALR(opt, swa_lr=SWA_LR_OPT) 
    
    best_swa_loss = float('inf')
    best_swa_state = None
    fold_t_loss, fold_v_loss = [], []
    
    for epoch in range(EPOCHS_RUN):
        model.train()
        t_loss = 0.0
        opt.zero_grad()
        
        for i, (inputs, labels) in enumerate(loader_tr):
            inputs, labels = inputs.to(dispositivo, dtype=torch.float32), labels.to(dispositivo).view(-1)
            
            with torch.cuda.amp.autocast():
                if (epoch < SWA_START) and (np.random.rand() < CUTMIX_PROB):
                    inputs, labels_a, labels_b, lam = cutmix_data(inputs, labels)
                    logits, _, _ = model(inputs)
                    loss = (lam * crit_coral(logits, get_cascade_labels(labels_a, dispositivo)) + 
                            (1 - lam) * crit_coral(logits, get_cascade_labels(labels_b, dispositivo))) / ACCUM_STEPS
                else:
                    logits, _, _ = model(inputs)
                    loss = crit_coral(logits, get_cascade_labels(labels, dispositivo)) / ACCUM_STEPS
                    
            scaler_amp.scale(loss).backward()
            if (i + 1) % ACCUM_STEPS == 0 or (i + 1) == len(loader_tr):
                scaler_amp.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler_amp.step(opt)
                scaler_amp.update()
                opt.zero_grad(set_to_none=True)
                
                if epoch >= SWA_START: swa_scheduler.step()
                else: scheduler.step()
            
            if not torch.isnan(loss): t_loss += (loss.item() / 2.0) * ACCUM_STEPS 
            
        # ==============================================================================
        # FASE DE VALIDACIÓN (MONITORIZACIÓN ORDINAL)
        # ==============================================================================
        model.eval()
        v_loss_total = 0.0
        v_preds_epoch, v_targets_epoch = [], []
        
        with torch.no_grad():
            for imgs, targets in loader_va:
                imgs, targets = imgs.to(dispositivo, dtype=torch.float32), targets.to(dispositivo).view(-1)
                with torch.cuda.amp.autocast():
                    logits, _, _ = model(imgs)
                    v_loss_total += (crit_coral(logits, get_cascade_labels(targets, dispositivo)).item() / 2.0)
                    
                    probs = torch.sigmoid(logits)
                    expected_y = 0.5 * probs[:, 0] + 0.5 * probs[:, 1]
                    
                    v_preds_epoch.extend(expected_y.cpu().numpy().flatten())
                    v_targets_epoch.extend(targets.cpu().numpy().flatten()) 
         
        fold_t_loss.append(t_loss / max(1, len(loader_tr)))
        fold_v_loss.append(v_loss_total / max(1, len(loader_va)))
        mae_ep = mean_absolute_error(v_targets_epoch, v_preds_epoch)

        estado_swa = f"(CutMix Activo)"
        
        # ==============================================================================
        # FASE SWA
        # ==============================================================================
        if epoch >= SWA_START:
            swa_model.update_parameters(model)
            swa_model.eval()
            swa_loss_val = 0.0
            swa_preds_epoch = []
            
            with torch.no_grad():
                for imgs, targets in loader_va:
                    imgs, targets_t = imgs.to(dispositivo, dtype=torch.float32), targets.to(dispositivo).view(-1)
                    with torch.cuda.amp.autocast(): 
                        logits, _, _ = swa_model(imgs)
                        swa_loss_val += crit_coral(logits, get_cascade_labels(targets_t, dispositivo)).item()
                        
                        probs = torch.sigmoid(logits)
                        expected_y = 0.5 * probs[:, 0] + 0.5 * probs[:, 1]
                        swa_preds_epoch.extend(expected_y.cpu().numpy().flatten())
                        
            swa_loss_val /= max(1, len(loader_va))
            swa_mae_ep = mean_absolute_error(v_targets_epoch, swa_preds_epoch)
            estado_swa = f"(SWA | Loss: {swa_loss_val:.4f} | MAE: {swa_mae_ep:.4f})"
            
            if swa_loss_val < best_swa_loss: 
                best_swa_loss = swa_loss_val
                best_swa_state = copy.deepcopy(swa_model.state_dict())
                
        print(f"  Ep {epoch+1:02d}/{EPOCHS_RUN} | Train L: {fold_t_loss[-1]:.4f} | Val L: {fold_v_loss[-1]:.4f} | Val MAE: {mae_ep:.4f} {estado_swa}")

    if best_swa_state is not None: 
        swa_model.load_state_dict(best_swa_state)

    historial_train_loss.append(fold_t_loss)
    historial_val_loss.append(fold_v_loss)
    
    ruta_npz = os.path.join(DIR_GUARDADO, 'resultados_visuales_oof.npz')
    if len(historial_train_loss) > 0 and len(historial_train_loss[0]) > 0:
        try:
            t_curve = np.mean(historial_train_loss, axis=0)
            v_curve = np.mean(historial_val_loss, axis=0)
            np.savez_compressed(ruta_npz, train_loss_curve=t_curve, val_loss_curve=v_curve)
        except Exception as e:
            print(f"    Aviso: No se pudo guardar la curva de pérdida ({e})")

    # ==============================================================================
    # INFERENCIA FINAL (TTA ORDINAL)
    # ==============================================================================
    def inferir_expected(loader):
        ps_all, ts_all = [], []
        for imgs, targets in loader:
            imgs = imgs.to(dispositivo); B, S, C, H, W = imgs.shape; swa_model.eval() 
            with torch.no_grad(), torch.cuda.amp.autocast():
                logits_base = swa_model(imgs)[0] 
                probs_det = [torch.sigmoid(logits_base.float())]
                
                for _ in range(N_TTA - 1): 
                    logits_tta = swa_model(tta_transforms_test(imgs.view(B*S, C, H, W)).view(B, S, C, H, W))[0]
                    probs_det.append(torch.sigmoid(logits_tta.float()))
                
                mean_probs = torch.stack(probs_det).mean(dim=0)
                expected_y = 0.5 * mean_probs[:, 0] + 0.5 * mean_probs[:, 1]
                
            ps_all.extend(expected_y.cpu().numpy().flatten()); ts_all.extend(targets.numpy().flatten())
        return np.array(ps_all), np.array(ts_all)

    print("    Calculando Inferencia TTA (Cascada Ordinal)...")
    v_preds_raw, v_targets = inferir_expected(loader_va)
    t_preds_raw, t_targets = inferir_expected(loader_te)
    
    try:
        fold_mae_test = mean_absolute_error(t_targets, t_preds_raw)
        fold_auc_test = roc_auc_score((t_targets > 0).astype(int), t_preds_raw)
        print(f"    TEST (Fold {fold_out + 1}) -> MAE: {fold_mae_test:.4f} | B-AUC (Sano vs Resto): {fold_auc_test:.4f}")
    except ValueError: pass
    
    resultados_visuales_por_fold[fold_out] = {
        'val_preds': v_preds_raw, 'val_targets': v_targets, 
        'test_preds': t_preds_raw, 'test_targets': t_targets, 
        'test_idx': np.array(test_idx), 'val_idx': np.array(df_train_val.iloc[val_idx_inner].index)
    }
    
    ruta_pesos = os.path.join(DIR_GUARDADO, f'swa_model_fold_{fold_out}.pth')
    torch.save(swa_model.module.state_dict(), ruta_pesos)
    
    del model, swa_model, loader_tr, loader_va, loader_te; gc.collect(); torch.cuda.empty_cache()

print(f"\n Entrenamiento Ordinal completado. Curvas y Pesos guardados.")

In [ ]:
# ==============================================================================
# CELDA 6: EVALUACIÓN ORDINAL SOTA (CALIBRACIÓN GLOBAL MAXIMIN)
# ==============================================================================

print("="*85)
print(" Régimen ordinal")
print("="*85)

# -------------------------------------------------------------------------
# 1. FUNCIONES MATEMÁTICAS Y DE EVALUACIÓN
# -------------------------------------------------------------------------
def optimizar_umbrales_maximin(y_true_cont, y_pred_cont):
    """
    Utiliza el principio Min-Max Fairness. 
    Maximiza el rendimiento de la peor clase (la que tiene menos aciertos).
    Esto garantiza matemáticamente la diagonal más equilibrada posible.
    """
    y_true_int = np.round(y_true_cont * 2).astype(int)
    best_score = -np.inf
    
    # Rango de búsqueda dinámico basado en percentiles
    candidatos = np.unique(np.percentile(y_pred_cont, np.linspace(5, 95, 100)))
    best_th = [np.percentile(y_pred_cont, 40), np.percentile(y_pred_cont, 80)]
    
    if len(candidatos) < 3: return best_th
    
    for i in range(len(candidatos)-1):
        for j in range(i+1, len(candidatos)):
            t1, t2 = candidatos[i], candidatos[j]
            
            y_p = np.zeros_like(y_pred_cont)
            y_p[(y_pred_cont > t1) & (y_pred_cont <= t2)] = 1
            y_p[y_pred_cont > t2] = 2
            
            cm = confusion_matrix(y_true_int, y_p, labels=[0, 1, 2])
            
            # Recalls de cada clase (Sensibilidad)
            recalls = np.diag(cm) / np.maximum(cm.sum(axis=1), 1)
            
            # La puntuación principal es el acierto de la PEOR clase
            peor_clase = np.min(recalls)
            # Desempate: Media global para decidir entre empates del peor caso
            media_clases = np.mean(recalls)
            
            score = peor_clase + (0.1 * media_clases)
            
            if score > best_score:
                best_score = score
                best_th = [t1, t2]
                
    if best_th[0] >= best_th[1]: best_th[1] = best_th[0] + 1e-5
    return best_th

def evaluar_omni_estadistico_libre(y_true_cont, y_pred_cont, umbrales_ord):
    if len(np.unique(y_true_cont)) < 2: return None
    
    y_true_int = np.round(y_true_cont * 2).astype(int)
    y_pred_int = np.zeros_like(y_pred_cont)
    y_pred_int[(y_pred_cont > umbrales_ord[0]) & (y_pred_cont <= umbrales_ord[1])] = 1
    y_pred_int[y_pred_cont > umbrales_ord[1]] = 2
    y_pred_int = y_pred_int.astype(int)
    
    qwk = cohen_kappa_score(y_true_int, y_pred_int, weights='quadratic')
    mae = mean_absolute_error(y_true_cont, y_pred_cont)
    
    y_true_bin = (y_true_cont > 0).astype(int)
    try: auc_bin = roc_auc_score(y_true_bin, y_pred_cont)
    except ValueError: auc_bin = 0.5
    
    fpr, tpr, thresholds = roc_curve(y_true_bin, y_pred_cont)
    prec_c, rec_c, _ = precision_recall_curve(y_true_bin, y_pred_cont)
    
    opt_idx = np.argmax(tpr - fpr)
    umbral_bin = thresholds[opt_idx]
    y_pred_bin = (y_pred_cont >= umbral_bin).astype(int)
    cm_bin = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])
    tn, fp, fn, tp = cm_bin.ravel()
    
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    esp = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    return {
        'QWK': qwk, 'MAE': mae, 'AUC (Sano vs Resto)': auc_bin, 'PR-AUC': auc(rec_c, prec_c),
        'Sensibilidad': sens, 'Especificidad': esp, 
        'y_true_cont': y_true_cont, 'y_pred_cont': y_pred_cont,
        'y_true': y_true_bin, 'y_prob': y_pred_cont, 
        'fpr': fpr, 'tpr': tpr, 'precision': prec_c, 'recall': rec_c,
        'cm_ord': confusion_matrix(y_true_int, y_pred_int, labels=[0, 1, 2]),
        'cm': cm_bin
    }

# -------------------------------------------------------------------------
# 2. PREPARACIÓN DE DATOS OOF Y ESCENARIOS
# -------------------------------------------------------------------------
df_pacientes['Pred_Visual_RM'] = np.nan
for fold_idx, data in resultados_visuales_por_fold.items():
    if isinstance(data, np.ndarray): data = data.item()
    df_pacientes.loc[data['test_idx'], 'Pred_Visual_RM'] = data['test_preds']

if 'M/F' not in df_pacientes.columns and 'df_demog' in globals() and 'M/F' in df_demog.columns:
    df_pacientes = pd.merge(df_pacientes, df_demog[['id_paciente', 'M/F']], left_on='id', right_on='id_paciente', how='left')

if 'M/F' in df_pacientes.columns:
    df_pacientes['M/F_bin'] = df_pacientes['M/F'].map({'M': 1, 'F': 0}).fillna(0)
    cols_base = ['Age', 'Educ', 'SES', 'eTIV', 'M/F_bin']
else:
    cols_base = ['Age', 'Educ', 'SES', 'eTIV'] 

escenarios = [
    ('1. Visual Aislada',               []),
    ('2. Tabular (Sin MMSE, Sin nWBV)', cols_base),
    ('3. Tabular (Sin MMSE)',           cols_base + ['nWBV']),
    ('4. Tabular (Completo)',           cols_base + ['nWBV', 'MMSE']),
    ('5. Fusión (Sin MMSE, Sin nWBV)',  cols_base + ['Pred_Visual_RM']),
    ('6. Fusión Profunda (Sin MMSE)',   cols_base + ['nWBV', 'Pred_Visual_RM']),
    ('7. Fusión Profunda (Completo)',   cols_base + ['nWBV', 'MMSE', 'Pred_Visual_RM'])
]

orden_modelos = [e[0] for e in escenarios]
almacen_resultados = {m: [] for m in orden_modelos}
registro_alphas = {m: [] for m in orden_modelos if 'Fusión' in m}

# -------------------------------------------------------------------------
# 3. MOTOR DE FUSIÓN Y CALIBRACIÓN GLOBAL
# -------------------------------------------------------------------------
for nombre_modelo, cols in escenarios:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    
    oof_y_true = []
    oof_y_pred = []
    fold_data = [] 
    
    for tr_idx, te_idx in skf.split(df_pacientes, df_pacientes['strat_dual']):
        df_tr, df_te = df_pacientes.iloc[tr_idx].copy(), df_pacientes.iloc[te_idx].copy()
        y_tr, y_te = df_tr['etiqueta'].values, df_te['etiqueta'].values
        
        if nombre_modelo == '1. Visual Aislada':
            preds_finales = df_te['Pred_Visual_RM'].values
        else:
            cols_clinicas = [c for c in cols if c != 'Pred_Visual_RM']
            pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), 
                             ('poly', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
                             ('model', BayesianRidge())])
            pipe.fit(df_tr[cols_clinicas], y_tr)
            preds_tab_te = np.clip(pipe.predict(df_te[cols_clinicas]), 0.0, 1.0)
            
            if 'Fusión' in nombre_modelo:
                preds_vis_tr = df_tr['Pred_Visual_RM'].values
                preds_tab_tr = np.clip(cross_val_predict(pipe, df_tr[cols_clinicas], y_tr, cv=5), 0.0, 1.0)
                
                best_s, best_a = float('inf'), 1.0
                for a in np.linspace(0, 1, 101):
                    p_fus = a * preds_vis_tr + (1-a) * preds_tab_tr
                    s = mean_absolute_error(y_tr, p_fus)
                    if s < best_s - 1e-4: 
                        best_s = s; best_a = a
                
                preds_finales = best_a * df_te['Pred_Visual_RM'].values + (1-best_a) * preds_tab_te
                registro_alphas[nombre_modelo].append(best_a)
            else:
                preds_finales = preds_tab_te
                
        oof_y_true.append(y_te)
        oof_y_pred.append(preds_finales)
        fold_data.append((y_te, preds_finales))

    y_true_all = np.concatenate(oof_y_true)
    y_pred_all = np.concatenate(oof_y_pred)
    best_th_global = optimizar_umbrales_maximin(y_true_all, y_pred_all)
    
    for y_te_fold, preds_te_fold in fold_data:
        res = evaluar_omni_estadistico_libre(y_te_fold, preds_te_fold, best_th_global)
        if res: almacen_resultados[nombre_modelo].append(res)

# -------------------------------------------------------------------------
# 4. IMPRESIÓN DE RESULTADOS
# -------------------------------------------------------------------------
def calc_ci(data):
    m = np.mean(data); err = st.sem(data)
    if err == 0 or len(data) < 2: return m, m, m, 0.0
    h = err * st.t.ppf((1 + 0.95) / 2., len(data)-1)
    return m, max(0, m-h), min(1, m+h), h

print("\n" + "="*85)
print(" REPORTES DE RENDIMIENTO ORDINAL: CALIBRACIÓN GLOBAL MAXIMIN")
print("="*85)

metricas_imprimir = ['QWK', 'MAE', 'AUC (Sano vs Resto)', 'PR-AUC', 'Sensibilidad', 'Especificidad']

for m in orden_modelos:
    print(f"\n MODELO: {m.upper()}")
    if m in registro_alphas: print(f"   [!] Alpha (Peso Media): {np.mean(registro_alphas[m]):.3f}")
    print("-" * 85)
    print(f"{'Métrica':<20} | {'Media':<10} | {'Lím. Inf.':<10} | {'Lím. Sup.':<10} | {'Variación':<10}")
    print("-" * 85)
    for met in metricas_imprimir:
        vals = [r[met] for r in almacen_resultados[m]]
        if vals:
            med, inf, sup, var = calc_ci(vals)
            print(f"{met:<20} | {med:<10.4f} | {inf:<10.4f} | {sup:<10.4f} | ±{var:<10.4f}")

In [ ]:
# ==============================================================================
# CELDA 7: BATERÍA GRÁFICA FINAL ORDINAL (SHAP, LIMPIA, COLOR NORMALIZADO)
# ==============================================================================

DIR_GRAFICOS = './graficos_tfg_final/'
os.makedirs(DIR_GRAFICOS, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
# Paleta ajustada a exactamente 6 colores
colores_modelos = ['#9b59b6', '#34495e', '#e74c3c', '#f39c12', '#1abc9c', '#3498db'] 

# -------------------------------------------------------------------------
# FILTRADO BLINDADO Y RENOMBRADO EXACTO (6 MODELOS)
# -------------------------------------------------------------------------
orden_filtrado = []
for m in orden_modelos:
    # Destruimos exclusivamente el modelo puramente tabular de demografía
    if 'Tabular' in m and ('Demografía' in m or 'Demografia' in m) and 'Sin MMSE' not in m and 'Volumetría' not in m:
        continue
    if m.startswith('2. Tabular'): # Filtro de seguridad
        continue
    orden_filtrado.append(m)

def obtener_etiqueta(nombre):
    """Asigna los nombres limpios solicitados asegurando el orden 1 al 6"""
    if '1.' in nombre or ('Visual' in nombre and 'Fusión' not in nombre): 
        return "1 Visual (ConvNeXt)"
    elif '3.' in nombre or ('Tabular' in nombre and 'Sin MMSE' in nombre): 
        return "2 Tabular (Sin MMSE)"
    elif '4.' in nombre or ('Tabular' in nombre and 'Completo' in nombre): 
        return "3 Tabular (Completo)"
    elif '5.' in nombre or ('Fusión' in nombre and 'Demografía' in nombre): 
        return "4 Fusión (Demografía)"
    elif '6.' in nombre or ('Fusión' in nombre and 'Sin MMSE' in nombre): 
        return "5 Fusión (Sin MMSE)"
    elif '7.' in nombre or ('Fusión' in nombre and 'Completo' in nombre): 
        return "6 Fusión (Completa)"
    return nombre

# -------------------------------------------------------------------------
# 1. EXPLICABILIDAD SHAP (Ridge Bayesiana Tabular)
# -------------------------------------------------------------------------
print("\n   Calculando interpretabilidad SHAP (Ridge Bayesiana Ordinal)...")
try:
    df_shap = df_pacientes.copy()
    
    if 'M/F' not in df_shap.columns and 'M/F' in df_demog.columns:
        df_shap = pd.merge(df_shap, df_demog[['id_paciente', 'M/F']], left_on='id', right_on='id_paciente', how='left')
    if 'M/F' in df_shap.columns:
        df_shap['M/F_bin'] = df_shap['M/F'].map({'M': 1, 'F': 0}).fillna(0)
        cols_tab_shap = ['Age', 'Educ', 'SES', 'eTIV', 'M/F_bin', 'MMSE', 'nWBV']
    else:
        cols_tab_shap = ['Age', 'Educ', 'SES', 'eTIV', 'MMSE', 'nWBV']

    # Extraemos la etiqueta ordinal continua (0.0, 0.5, 1.0)
    if 'etiqueta_continua' in df_shap.columns:
        y_true_shap = df_shap['etiqueta_continua'].values
    elif 'CDR' in df_shap.columns:
        y_true_shap = df_shap['CDR'].astype(float).values
    else:
        y_true_shap = df_shap['etiqueta'].values

    # Imputación y Escalado
    imputer_shap = KNNImputer(n_neighbors=5)
    scaler_shap = StandardScaler()
    X_tab_raw = imputer_shap.fit_transform(df_shap[cols_tab_shap])
    X_tab_sc = scaler_shap.fit_transform(X_tab_raw)

    # Entrenar modelo tabular ordinal (Bayesian Ridge)
    modelo_tabular_shap = BayesianRidge()
    modelo_tabular_shap.fit(X_tab_sc, y_true_shap)

    # Calcular SHAP Values 
    explainer_tab = shap.LinearExplainer(modelo_tabular_shap, X_tab_sc)
    shap_values_tab = explainer_tab.shap_values(X_tab_sc)

    # Gráfico SHAP Tabular
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values_tab, X_tab_sc, feature_names=cols_tab_shap, plot_type="dot", show=False)
    plt.title('SHAP: Impacto de Variables Clínicas (Ridge Bayesiana Ordinal)', fontweight='bold', fontsize=16, y=1.05)
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_GRAFICOS, '01_SHAP_Tabular_Ordinal.png'), dpi=300, bbox_inches='tight')
    plt.show(); plt.close()
    print(" ✓ Gráfico SHAP Tabular Ordinal generado.")

except Exception as e:
    print(f" Aviso: No se pudo generar el gráfico SHAP ({e}). Asegúrate de tener instalada la librería 'shap'.")

 
# -------------------------------------------------------------------------
# 2. CURVAS DE APRENDIZAJE
# -------------------------------------------------------------------------
try:
    ruta_npz = os.path.join('./modelos_entrenados/', 'resultados_visuales_oof.npz')
    datos_npz = np.load(ruta_npz, allow_pickle=True)
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(datos_npz['train_loss_curve']) + 1), datos_npz['train_loss_curve'], label='Train Loss', color='#2980b9', lw=2.5)
    plt.plot(range(1, len(datos_npz['val_loss_curve']) + 1), datos_npz['val_loss_curve'], label='Val Loss', color='#c0392b', lw=2.5)
    plt.title('Curva de Aprendizaje Visual (CORAL)', fontweight='bold', fontsize=16)
    plt.xlabel('Épocas'); plt.ylabel('Pérdida (CORAL Loss)'); plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(DIR_GRAFICOS, '02_Curva_Loss.png'), dpi=300); plt.show(); plt.close()
except Exception as e: 
    print(f" Aviso: Loss no disponible ({e}).")

# -------------------------------------------------------------------------
# 3. CURVAS ROC (CON AUC EN LEYENDA)
# -------------------------------------------------------------------------
plt.figure(figsize=(10, 8))
mean_fpr = np.linspace(0, 1, 100)
for idx, nombre in enumerate(orden_filtrado):
    tprs, aucs = [], []
    for fold_data in almacen_resultados[nombre]:
        if 'fpr' in fold_data:
            tprs.append(np.interp(mean_fpr, fold_data['fpr'], fold_data['tpr']))
            aucs.append(fold_data.get('AUC (Sano vs Resto)', 0.5))
    if tprs:
        mean_tpr = np.mean(tprs, axis=0); mean_tpr[0] = 0.0; mean_tpr[-1] = 1.0
        mean_auc = np.mean(aucs)
        etiqueta_limpia = obtener_etiqueta(nombre)
        plt.plot(mean_fpr, mean_tpr, color=colores_modelos[idx % len(colores_modelos)], lw=2.5, label=f'{etiqueta_limpia} (AUC = {mean_auc:.3f})')

plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='k', label='Línea de Azar')
plt.title('Comparativa de Curvas ROC (Sano vs Resto)', fontweight='bold', fontsize=16)
plt.xlabel('1 - Especificidad (FPR)'); plt.ylabel('Sensibilidad (TPR)'); plt.legend(loc="lower right", fontsize='small')
plt.tight_layout(); plt.savefig(os.path.join(DIR_GRAFICOS, '03_ROC_Filtradas.png'), dpi=300); plt.show(); plt.close()

# -------------------------------------------------------------------------
# 4. MATRICES DE CONFUSIÓN (3x3) - COLOR NORMALIZADO Y GRID AJUSTADO (2x3)
# -------------------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(18, 11)) 
fig.suptitle('Matrices de Confusión Ordinales (3x3)', fontsize=22, fontweight='bold', y=1.02)
axes = axes.flatten()

for i, nombre in enumerate(orden_filtrado):
    cm_global = np.sum([r['cm_ord'] for r in almacen_resultados[nombre]], axis=0)
    
    # Normalizamos SOLO el color para que la diagonal de AD no se vea pálida
    cm_norm = cm_global.astype('float') / cm_global.sum(axis=1)[:, np.newaxis]
    cm_norm = np.nan_to_num(cm_norm) 
    
    sns.heatmap(cm_norm, annot=cm_global, fmt='d', cmap='Purples', ax=axes[i], cbar=False, 
                annot_kws={"size": 16, "weight": "bold"}, vmin=0, vmax=1)
    
    etiqueta_limpia = obtener_etiqueta(nombre)
    axes[i].set_title(etiqueta_limpia, fontsize=14, fontweight='bold')
    axes[i].set_xticklabels(['Sano', 'MCI', 'AD'], fontsize=11)
    axes[i].set_yticklabels(['Sano', 'MCI', 'AD'], fontsize=11, rotation=0)

plt.tight_layout(rect=[0, 0, 1, 0.96]); plt.savefig(os.path.join(DIR_GRAFICOS, '04_Matrices_Limpias.png'), dpi=300); plt.show(); plt.close()

# -------------------------------------------------------------------------
# 5. DISTRIBUCIÓN KDE (SEPARABILIDAD TOPOLÓGICA ORDINAL) - PARA LOS 6 MODELOS
# -------------------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(18, 11)) 
fig.suptitle('KDE - Separabilidad de Clases (Sano, MCI, Alzheimer)', fontsize=22, fontweight='bold', y=1.02)
axes = axes.flatten()

for i, nombre in enumerate(orden_filtrado):
    y_t = np.concatenate([r['y_true_cont'] for r in almacen_resultados[nombre]])
    y_p = np.concatenate([r['y_pred_cont'] for r in almacen_resultados[nombre]])
    
    sns.kdeplot(y_p[y_t == 0.0], color='#3498db', fill=True, alpha=0.5, lw=2, ax=axes[i], label='Sano')
    sns.kdeplot(y_p[y_t == 0.5], color='#f39c12', fill=True, alpha=0.5, lw=2, ax=axes[i], label='MCI')
    sns.kdeplot(y_p[y_t == 1.0], color='#e74c3c', fill=True, alpha=0.5, lw=2, ax=axes[i], label='Alzheimer')
    
    etiqueta_limpia = obtener_etiqueta(nombre)
    axes[i].set_title(etiqueta_limpia, fontsize=14, fontweight='bold')
    axes[i].set_xlim(-0.05, 1.05)
    axes[i].set_xlabel('Riesgo Estimado Continuo'); axes[i].set_ylabel('Densidad')
    if i == 0: axes[i].legend(loc='upper right', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(DIR_GRAFICOS, '05_KDE_Todos_Modelos.png'), dpi=300, bbox_inches='tight')
plt.show(); plt.close()

# -------------------------------------------------------------------------
# 6. CURVAS PRECISION-RECALL (CON PR-AUC EN LEYENDA)
# -------------------------------------------------------------------------
plt.figure(figsize=(10, 8))
mean_recall = np.linspace(0, 1, 100)
for idx, nombre in enumerate(orden_filtrado):
    precs, pr_aucs = [], []
    for fold_data in almacen_resultados[nombre]:
        if 'recall' in fold_data:
            precs.append(np.interp(mean_recall, fold_data['recall'][::-1], fold_data['precision'][::-1]))
            pr_aucs.append(fold_data.get('PR-AUC', 0.5))
    if precs:
        mean_prec = np.mean(precs, axis=0)
        mean_pr_auc = np.mean(pr_aucs)
        etiqueta_limpia = obtener_etiqueta(nombre)
        plt.plot(mean_recall, mean_prec, color=colores_modelos[idx % len(colores_modelos)], lw=2.5, label=f'{etiqueta_limpia} (PR-AUC = {mean_pr_auc:.3f})')

plt.title('Curvas Precision-Recall (Sano vs Resto)', fontweight='bold', fontsize=16)
plt.xlabel('Recall (Sensibilidad)'); plt.ylabel('Precisión (VPP)'); plt.legend(loc="lower left", fontsize='small')
plt.tight_layout(); plt.savefig(os.path.join(DIR_GRAFICOS, '06_PR_Filtradas.png'), dpi=300); plt.show(); plt.close()

# -------------------------------------------------------------------------
# 7. BOXPLOTS DE QWK Y MAE (ESTABILIDAD ORDINAL MULTICLASE)
# -------------------------------------------------------------------------
print("   Generando Boxplots de métricas ordinales (MAE y QWK)...")
datos_ordinales = []

# Extraemos las métricas SOLO para los 6 modelos filtrados
for m in orden_filtrado:
    etiqueta_limpia = obtener_etiqueta(m)
    for fold_data in almacen_resultados[m]:
        qwk_val = fold_data.get('QWK', np.nan)
        mae_val = fold_data.get('MAE', np.nan)
        
        if not np.isnan(qwk_val) and not np.isnan(mae_val):
            datos_ordinales.append({'Modelo': etiqueta_limpia, 'QWK': qwk_val, 'MAE': mae_val})

df_ordinal = pd.DataFrame(datos_ordinales)

if not df_ordinal.empty:
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle('Estabilidad de Métricas Ordinales en Validación Cruzada', fontsize=20, fontweight='bold', y=1.05)
    
    # 7A. Gráfico MAE (El objetivo es MINIMIZAR)
    sns.boxplot(x='Modelo', y='MAE', data=df_ordinal, palette=colores_modelos, ax=axes[0], width=0.6)
    sns.stripplot(x='Modelo', y='MAE', data=df_ordinal, color='black', alpha=0.5, ax=axes[0], size=5)
    axes[0].set_title('Error Absoluto Medio (MAE) ↓', fontweight='bold', fontsize=16)
    axes[0].tick_params(axis='x', rotation=25)
    axes[0].set_ylabel('MAE Continuo')
    axes[0].set_xlabel('')

    # 7B. Gráfico QWK (El objetivo es MAXIMIZAR)
    sns.boxplot(x='Modelo', y='QWK', data=df_ordinal, palette=colores_modelos, ax=axes[1], width=0.6)
    sns.stripplot(x='Modelo', y='QWK', data=df_ordinal, color='black', alpha=0.5, ax=axes[1], size=5)
    axes[1].set_title('Quadratic Weighted Kappa (QWK) ↑', fontweight='bold', fontsize=16)
    axes[1].tick_params(axis='x', rotation=25)
    axes[1].set_ylabel('Índice QWK')
    axes[1].set_xlabel('')

    plt.tight_layout()
    plt.savefig(os.path.join(DIR_GRAFICOS, '07_Boxplots_MAE_QWK.png'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
else:
    print("   Aviso: No se encontraron las claves 'QWK' o 'MAE'.")


In [ ]:
# ==============================================================================
# CELDA 8: INTERPRETABILIDAD ORDINAL (GRAD-CAM 2x4 CON PARCHE DE GRADIENTES)
# ==============================================================================


print(" Iniciando grad-cam")
DIR_GRAFICOS = './graficos_tfg_libre/'
os.makedirs(DIR_GRAFICOS, exist_ok=True)

# Parámetros
FILAS = 2
COLUMNAS = 4
NUM_PACIENTES = FILAS * COLUMNAS  
TOP_K_CORTES = 1  

# 1. Recuperación del Modelo Ordinal
modelo_xai = HybridAgile_MIDL().to(dispositivo)
ruta_pesos = os.path.join(DIR_GUARDADO, 'swa_model_fold_0.pth')

try:
    modelo_xai.load_state_dict(torch.load(ruta_pesos, map_location=dispositivo))
    print("Pesos SWA cargados correctamente desde disco.")
except FileNotFoundError:
    print("No se hallaron pesos entrenados. Usa pesos aleatorios para debug.")

modelo_xai.eval()


def forward_xai_patch(self, x):
    B, S, C, H, W = x.shape
    x = x.view(B * S, C, H, W)
    
    features_spatial = self.backbone.features(x) 
    
    features_flat = self.adaptive_pool(self.fine_grained_extractor(features_spatial)).view(B * S, -1)
    A = F.softmax(self.attention_weights(self.attention_V(features_flat) * self.attention_U(features_flat)).view(B, S), dim=1)
    features_flat = features_flat.view(B, S, -1)
    M = torch.bmm(A.unsqueeze(1), features_flat).squeeze(1)
    latent_128d = self.latent_extractor(M)
    
    proj = self.fc(latent_128d)             
    logits = proj + self.ordinal_bias       
    return logits, latent_128d, A

# Sobreescribimos el método en la instancia cargada en memoria
modelo_xai.forward = types.MethodType(forward_xai_patch, modelo_xai)
# =========================================================================

# 2. SELECCIÓN ESTOCÁSTICA DE PACIENTES CON DEMENCIA (CDR >= 1)
df_ad = df_pacientes[df_pacientes['etiqueta'] >= 1.0].copy()
num_imagenes_reales = min(NUM_PACIENTES, len(df_ad))
df_ad_sample = df_ad.sample(n=num_imagenes_reales).reset_index(drop=True)

ds_xai = OASIS_Visual_Dataset(df_ad_sample, is_train=False, n_cortes=N_CORTES)
loader_xai = DataLoader(ds_xai, batch_size=1, num_workers=0, shuffle=False)

# 3. Preparación de los Hooks  
activaciones, gradientes = {}, {}
def forward_hook(m, i, o): activaciones['value'] = o
def backward_hook(m, gi, go): gradientes['value'] = go[0]

capa_objetivo = modelo_xai.backbone.features[-1]
handle_fw = capa_objetivo.register_forward_hook(forward_hook)
handle_bw = capa_objetivo.register_full_backward_hook(backward_hook)

overlays_guardados = []
titulos_guardados = []

# 4. Extracción de Mapas Térmicos Ordinales
for idx, (imgs, targets) in enumerate(loader_xai):
    input_tensor = imgs.to(dispositivo)
    # Forzamos a que el tensor de entrada rastree el gradiente
    input_tensor.requires_grad_(True)
    
    paciente_id = df_ad_sample.iloc[idx]['id']
    
    modelo_xai.zero_grad()
    logits, latents, attention = modelo_xai(input_tensor)
    
    # Retropropagación de la Esperanza Matemática E[y] (Sub-tarea Ordinal)
    probs = torch.sigmoid(logits)
    esperanza_y = 0.5 * probs[0, 0] + 0.5 * probs[0, 1] 
    esperanza_y.backward(retain_graph=False) 
    
    feats = activaciones['value'].detach()
    grads = gradientes['value'].detach()
    
    pesos_canales = torch.mean(grads, dim=[2, 3], keepdim=True) 
    cams = torch.sum(pesos_canales * feats, dim=1) 
    cams = F.relu(cams) 
    
    pesos_atencion = attention[0].cpu().detach().numpy()
    
    # Extraer el Top 1 de Atención (Corte más patológico)
    top_k_indices = np.argsort(pesos_atencion)[-TOP_K_CORTES:][::-1]
    
    for rank, corte_idx in enumerate(top_k_indices):
        cam_actual = cams[corte_idx].cpu().numpy()
        cam_actual = cv2.resize(cam_actual, (224, 224))
        
        if np.max(cam_actual) - np.min(cam_actual) > 0:
            cam_actual = (cam_actual - np.min(cam_actual)) / (np.max(cam_actual) - np.min(cam_actual))
        else:
            cam_actual = np.zeros_like(cam_actual)
        
        img_original = input_tensor[0, corte_idx].detach().cpu().numpy().transpose(1, 2, 0)
        img_original = (img_original - img_original.min()) / (img_original.max() - img_original.min() + 1e-8)
        
        heatmap = cv2.applyColorMap(np.uint8(255 * cam_actual), cv2.COLORMAP_JET)
        heatmap = np.float32(heatmap) / 255
        heatmap = heatmap[:, :, ::-1] 
        
        cam_overlay = heatmap * 0.4 + img_original * 0.6
        cam_overlay = cam_overlay / np.max(cam_overlay)
        
        overlays_guardados.append(cam_overlay)
        titulos_guardados.append(f"ID: {paciente_id}\n$\mathbb{{E}}[y]$: {esperanza_y.item():.2f} | Atn: {pesos_atencion[corte_idx]:.2f}")

handle_fw.remove()
handle_bw.remove()

# 5. DIBUJO DE LA CUADRÍCULA (GRID 2x4 ESTRICTO)
fig, axes = plt.subplots(FILAS, COLUMNAS, figsize=(22, 11))
fig.suptitle('8. Focos de Neurodegeneración Ordinal (Esperanza Matemática $\mathbb{E}[y]$)', 
             fontweight='bold', fontsize=20, y=1.02)

axes_flat = axes.flatten()

for i, ax in enumerate(axes_flat):
    if i < len(overlays_guardados):
        ax.imshow(overlays_guardados[i])
        ax.set_title(titulos_guardados[i], fontsize=13, fontweight='bold', pad=8)
    ax.axis('off')

plt.tight_layout()
ruta_guardado = os.path.join(DIR_GRAFICOS, 'Grad_CAM_Ordinal_2x4.png')
plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
plt.show()

 